In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    预处理文本，构建词汇表，生成滑动窗口特征和标签
    
    Args:
        text: 输入文本字符串
        n: 窗口大小（特征序列长度）
    
    Returns:
        vocab: 词汇表字典 {词: ID}
        features: 特征列表
        labels: 标签列表
    """
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    tokens = text.split()
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    token_counts = Counter(tokens)
    # 按频率降序排序，频率相同则按字母顺序
    sorted_tokens = sorted(token_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {token: idx for idx, (token, _) in enumerate(sorted_tokens)}
    
    # 4. 用滑动窗口生成长度为n的特征序列和对应的下一个词标签
    features = []
    labels = []
    
    for i in range(len(tokens) - n):
        features.append(tokens[i:i+n])
        labels.append(tokens[i+n])
    
    return vocab, (features, labels)


# 测试
if __name__ == "__main__":
    text = "The time machine"
    n = 2
    vocab, (features, labels) = preprocess_text(text, n)
    
    print("词汇表:", vocab)
    print("特征列表:", features)
    print("标签列表:", labels)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征列表: [['the', 'time']]
标签列表: ['machine']


In [2]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单元前向传播
    
    Args:
        x_t: 输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        W_hx: 输入权重，形状 (input_size, hidden_size)
        W_hh: 隐藏权重，形状 (hidden_size, hidden_size)
        b_h: 偏置，形状 (hidden_size,)
    
    Returns:
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        cache: 缓存用于反向传播
    """
    # 计算候选隐藏状态
    h_t = np.tanh(np.dot(x_t, W_hx) + np.dot(h_prev, W_hh) + b_h)
    
    cache = (x_t, h_prev, h_t, W_hx, W_hh, b_h)
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    """
    RNN单元反向传播
    
    Args:
        dh_next: 上游梯度，形状 (batch_size, hidden_size)
        cache: 前向传播缓存
    
    Returns:
        dx_t: 输入梯度，形状 (batch_size, input_size)
        dh_prev: 隐藏状态梯度，形状 (batch_size, hidden_size)
        dW_hx: 输入权重梯度，形状 (input_size, hidden_size)
        dW_hh: 隐藏权重梯度，形状 (hidden_size, hidden_size)
        db_h: 偏置梯度，形状 (hidden_size,)
    """
    x_t, h_prev, h_t, W_hx, W_hh, b_h = cache
    
    batch_size = x_t.shape[0]
    
    # tanh的导数: d/dx tanh(x) = 1 - tanh(x)^2
    dh = dh_next * (1 - h_t**2)
    
    # 计算梯度
    dx_t = np.dot(dh, W_hx.T)
    dh_prev = np.dot(dh, W_hh.T)
    dW_hx = np.dot(x_t.T, dh)
    dW_hh = np.dot(h_prev.T, dh)
    db_h = np.sum(dh, axis=0)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# 测试
if __name__ == "__main__":
    np.random.seed(42)
    
    batch_size = 2
    input_size = 4
    hidden_size = 3
    
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hx = np.random.randn(input_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    b_h = np.random.randn(hidden_size)
    
    # 前向传播
    h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
    print("前向传播结果 h_t:")
    print(h_t)
    
    # 反向传播
    dh_next = np.random.randn(batch_size, hidden_size)
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)
    
    print("\n反向传播梯度:")
    print(f"dx_t shape: {dx_t.shape}")
    print(f"dh_prev shape: {dh_prev.shape}")
    print(f"dW_hx shape: {dW_hx.shape}")
    print(f"dW_hh shape: {dW_hh.shape}")
    print(f"db_h shape: {db_h.shape}")

前向传播结果 h_t:
[[-0.99457277 -0.73194763 -0.8174345 ]
 [ 0.67612543  0.9018296  -0.96713191]]

反向传播梯度:
dx_t shape: (2, 4)
dh_prev shape: (2, 3)
dW_hx shape: (4, 3)
dW_hh shape: (3, 3)
db_h shape: (3,)


In [3]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    """
    双向RNN编码器
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 使用PyTorch的RNN实现双向
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=False,
            bidirectional=True
        )
    
    def forward(self, X):
        """
        前向传播
        
        Args:
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        Returns:
            outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
            final_state: 最终时间步的拼接隐藏状态，形状 (2*hidden_dim,)
        """
        # RNN前向传播
        # outputs: (seq_len, batch, 2*hidden_dim)
        # h_n: (2*num_layers, batch, hidden_dim)
        outputs, h_n = self.rnn(X)
        
        # 获取最终时间步的隐藏状态
        # 取最后一层的前向和后向状态
        last_layer_forward = h_n[-2, :, :]   # 前向
        last_layer_backward = h_n[-1, :, :]  # 后向
        final_state = torch.cat([last_layer_forward, last_layer_backward], dim=1)
        
        return outputs, final_state


# 使用PyTorch RNN的另一种实现（手动）
class BidirectionalRNNEncoderManual(nn.Module):
    """
    手动实现的双向RNN编码器
    """
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # 前向RNN
        self.rnn_forward = nn.RNNCell(input_dim, hidden_dim)
        # 后向RNN
        self.rnn_backward = nn.RNNCell(input_dim, hidden_dim)
    
    def forward(self, X):
        """
        前向传播
        
        Args:
            X: 输入序列，形状 (seq_len, batch, input_dim)
        """
        seq_len, batch, _ = X.shape
        
        # 初始化隐藏状态
        h_f = torch.zeros(batch, self.hidden_dim)
        h_b = torch.zeros(batch, self.hidden_dim)
        
        # 存储所有时间步的隐藏状态
        forward_hidden = []
        backward_hidden = []
        
        # 前向传播
        for t in range(seq_len):
            h_f = self.rnn_forward(X[t], h_f)
            forward_hidden.append(h_f)
        
        # 后向传播
        for t in range(seq_len - 1, -1, -1):
            h_b = self.rnn_backward(X[t], h_b)
            backward_hidden.insert(0, h_b)
        
        # 拼接前向和后向隐藏状态
        outputs = []
        for t in range(seq_len):
            concat = torch.cat([forward_hidden[t], backward_hidden[t]], dim=1)
            outputs.append(concat)
        
        outputs = torch.stack(outputs)
        final_state = outputs[-1]  # 最终时间步的拼接状态
        
        return outputs, final_state


# 测试
if __name__ == "__main__":
    seq_len = 5
    batch = 3
    input_dim = 4
    hidden_dim = 6
    
    X = torch.randn(seq_len, batch, input_dim)
    
    # 使用PyTorch内置实现
    encoder = BidirectionalRNNEncoder(input_dim, hidden_dim)
    outputs, final_state = encoder(X)
    print("使用PyTorch RNN:")
    print(f"Outputs shape: {outputs.shape}")
    print(f"Final state shape: {final_state.shape}")
    
    # 使用手动实现
    encoder_manual = BidirectionalRNNEncoderManual(input_dim, hidden_dim)
    outputs_manual, final_state_manual = encoder_manual(X)
    print("\n使用手动实现:")
    print(f"Outputs shape: {outputs_manual.shape}")
    print(f"Final state shape: {final_state_manual.shape}")

使用PyTorch RNN:
Outputs shape: torch.Size([5, 3, 12])
Final state shape: torch.Size([3, 12])

使用手动实现:
Outputs shape: torch.Size([5, 3, 12])
Final state shape: torch.Size([3, 12])


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def cbow_forward(context_indices, W, W_out, target_indices=None):
    """
    CBOW模型前向传播和损失计算
    
    Args:
        context_indices: 上下文词索引列表，形状 (batch_size, context_size)
        W: 输入权重矩阵，形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
        target_indices: 中心词索引，形状 (batch_size,)，默认为None
    
    Returns:
        loss: 交叉熵损失（如果target_indices不为None）
        probs: 输出概率分布
    """
    # 获取上下文词的嵌入向量
    # context_indices: (batch_size, context_size)
    # W[context_indices]: (batch_size, context_size, d)
    context_embeddings = W[context_indices]  # (batch_size, context_size, d)
    
    # 计算平均上下文向量作为隐藏层
    # hidden: (batch_size, d)
    hidden = torch.mean(context_embeddings, dim=1)
    
    # 计算输出分数
    # scores: (batch_size, V)
    scores = torch.matmul(hidden, W_out)
    
    # 计算概率分布（softmax）
    probs = F.softmax(scores, dim=1)
    
    # 如果提供了目标索引，计算交叉熵损失
    if target_indices is not None:
        loss = F.cross_entropy(scores, target_indices)
        return loss, probs
    
    return probs


# 测试
if __name__ == "__main__":
    # 设置参数
    V = 10  # 词汇表大小
    d = 4   # 嵌入维度
    batch_size = 3
    context_size = 2
    
    # 初始化权重（需要梯度）
    W = torch.randn(V, d, requires_grad=True)
    W_out = torch.randn(d, V, requires_grad=True)
    
    # 生成随机数据
    context_indices = torch.randint(0, V, (batch_size, context_size))
    target_indices = torch.randint(0, V, (batch_size,))
    
    print("上下文索引:", context_indices)
    print("目标索引:", target_indices)
    
    # 前向传播
    loss, probs = cbow_forward(context_indices, W, W_out, target_indices)
    
    print(f"\n损失: {loss.item():.4f}")
    print(f"概率分布形状: {probs.shape}")
    print("概率分布示例（第一个样本）:", probs[0].detach().numpy()[:5])
    
    # 反向传播（梯度检查）
    loss.backward()
    print(f"\nW梯度形状: {W.grad.shape}")
    print(f"W_out梯度形状: {W_out.grad.shape}")
    
    # 验证梯度是否计算
    print(f"W梯度是否有值: {W.grad is not None}")
    print(f"W梯度范数: {W.grad.norm().item():.6f}")

上下文索引: tensor([[8, 4],
        [1, 3],
        [9, 9]])
目标索引: tensor([2, 4, 9])

损失: 3.6687
概率分布形状: torch.Size([3, 10])
概率分布示例（第一个样本）: [0.1999358  0.21063195 0.09863035 0.00174279 0.00706477]

W梯度形状: torch.Size([10, 4])
W_out梯度形状: torch.Size([4, 10])
W梯度是否有值: True
W梯度范数: 0.989043


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    """
    多头注意力机制
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, X):
        """
        前向传播
        
        Args:
            X: 输入，形状 (seq_len, batch, d_model)
        
        Returns:
            output: 输出，形状 (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape
        
        # 线性投影得到Q, K, V
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)  # (seq_len, batch, d_model)
        V = self.W_v(X)  # (seq_len, batch, d_model)
        
        # 重塑为多头形式
        # (seq_len, batch, num_heads, d_k) -> (seq_len, batch*num_heads, d_k)
        Q = Q.view(seq_len, batch * self.num_heads, self.d_k)
        K = K.view(seq_len, batch * self.num_heads, self.d_k)
        V = V.view(seq_len, batch * self.num_heads, self.d_v)
        
        # 缩放点积注意力
        # 转置为 (batch*num_heads, seq_len, d_k) 方便计算
        Q = Q.transpose(0, 1)
        K = K.transpose(0, 1)
        V = V.transpose(0, 1)
        
        # 计算注意力分数
        # scores: (batch*num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # softmax
        attn_weights = F.softmax(scores, dim=-1)
        
        # 加权求和
        # context: (batch*num_heads, seq_len, d_v)
        context = torch.matmul(attn_weights, V)
        
        # 重塑回原始形状
        # (batch*num_heads, seq_len, d_v) -> (seq_len, batch, d_model)
        context = context.transpose(0, 1).contiguous()
        context = context.view(seq_len, batch, self.d_model)
        
        # 最终线性层
        output = self.W_o(context)
        
        return output


# 简化版本（不使用nn.Linear，便于理解）
class MultiHeadAttentionSimple:
    def __init__(self, d_model, num_heads):
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
    
    def forward(self, X, W_q, W_k, W_v, W_o):
        """
        简化版前向传播
        
        Args:
            X: (seq_len, batch, d_model)
            W_q, W_k, W_v, W_o: 权重矩阵
        """
        seq_len, batch, _ = X.shape
        
        # 线性投影
        Q = torch.matmul(X, W_q)  # (seq_len, batch, d_model)
        K = torch.matmul(X, W_k)
        V = torch.matmul(X, W_v)
        
        # 重塑为多头
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k)
        K = K.view(seq_len, batch, self.num_heads, self.d_k)
        V = V.view(seq_len, batch, self.num_heads, self.d_v)
        
        # 重新排列维度
        Q = Q.permute(1, 2, 0, 3)  # (batch, num_heads, seq_len, d_k)
        K = K.permute(1, 2, 0, 3)
        V = V.permute(1, 2, 0, 3)
        
        # 计算注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, V)
        
        # 拼接所有头
        context = context.permute(2, 0, 1, 3)  # (seq_len, batch, num_heads, d_v)
        context = context.reshape(seq_len, batch, self.d_model)
        
        # 最终线性变换
        output = torch.matmul(context, W_o)
        
        return output


# 测试
if __name__ == "__main__":
    seq_len = 4
    batch = 2
    d_model = 4
    num_heads = 2
    
    X = torch.randn(seq_len, batch, d_model)
    
    # 使用PyTorch实现
    mha = MultiHeadAttention(d_model, num_heads)
    output = mha(X)
    
    print("Multi-Head Attention输出:")
    print(f"输入形状: {X.shape}")
    print(f"输出形状: {output.shape}")
    print("\n输出示例（第一个时间步，第一个样本）:")
    print(output[0, 0].detach().numpy())

Multi-Head Attention输出:
输入形状: torch.Size([4, 2, 4])
输出形状: torch.Size([4, 2, 4])

输出示例（第一个时间步，第一个样本）:
[-0.03035152  0.09393077  0.03378978  0.02388211]
